# Reproduce the final UltraFeedback results
Run all cells on a CPU. The notebook reads the 500 frozen prompts and recorded judgments from the repository, recomputes the tables and figures, and compares the results with the published CSVs. From a local clone, install the packages in `requirements.txt` first. In Colab, the first cell fetches the study files from the public repository.

In [ ]:
from pathlib import Path
import subprocess

# A local checkout works offline. A fresh Colab runtime checks out the final study from GitHub.
cwd = Path.cwd().resolve()
study_path = Path('ultrafeedback/final')
candidates = [p for parent in (cwd, *cwd.parents) for p in (parent, parent / study_path)]
root = next((p for p in candidates if (p / 'results/latest_decisions.csv').is_file()
             and (p / 'reproduce.py').is_file()), None)
if root is None:
    checkout = cwd / 'judge-bias-qwen3'
    if not (checkout / '.git').is_dir():
        subprocess.run(['git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
                        'https://github.com/shashank257/judge-bias-qwen3.git', str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'sparse-checkout', 'set', '--no-cone',
                    '/ultrafeedback/final/*', '!/ultrafeedback/final/UltraFeedback_Final.zip'], check=True)
    root = checkout / study_path
print('Using', root)

In [ ]:
import runpy
from IPython.display import Image, display

analysis = runpy.run_path(str(root / 'reproduce.py'))
display(analysis['replay']())
display(Image(filename=str(root / 'reproduced/pooled_drift.png')))